# EbbRAG — Evaluation (SPEC-03)

Experiment runner, metrics, checkpointing, and results aggregation.

**Requires:** Core classes from SPEC-01 and baselines from SPEC-02  
**Output:** `results/raw.jsonl` — one line per (system, query)

## 1. Imports

In [ ]:
!pip install -q scipy pandas matplotlib

import json, os, time
from pathlib import Path
from typing import Optional, List, Dict
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')  # Colab-safe backend
import matplotlib.pyplot as plt

os.makedirs('results', exist_ok=True)
print('Imports ✓')

## 2. Metrics

In [ ]:
def normalize(text: str) -> str:
    """Lowercase, strip punctuation and whitespace."""
    import re, string
    text = text.lower().strip()
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    return ' '.join(text.split())

def exact_match(prediction: str, ground_truth: str) -> int:
    """1 if ground truth appears in prediction (normalized), else 0."""
    return int(normalize(ground_truth) in normalize(prediction))

def token_f1(prediction: str, ground_truth: str) -> float:
    """Token-level F1 between prediction and ground truth."""
    pred_tokens = normalize(prediction).split()
    gt_tokens   = normalize(ground_truth).split()
    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall    = len(common) / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

def recall_at_k(retrieved_ids: List[str], relevant_ids: List[str]) -> int:
    """1 if any relevant chunk is in retrieved set."""
    return int(bool(set(retrieved_ids) & set(relevant_ids)))

def precision_at_k(retrieved_ids: List[str], relevant_ids: List[str]) -> float:
    """Fraction of retrieved chunks that are relevant."""
    if not retrieved_ids:
        return 0.0
    return len(set(retrieved_ids) & set(relevant_ids)) / len(retrieved_ids)

# Unit tests
assert exact_match('Jane Austen wrote it.', 'Jane Austen') == 1
assert exact_match('Charlotte Bronte wrote it.', 'Jane Austen') == 0
assert token_f1('the cat sat', 'the cat sat') == 1.0
assert token_f1('the dog ran', 'the cat sat') == 0.0
print('Metrics ✓')

## 3. ExperimentRunner with JSONL checkpointing

In [ ]:
class ExperimentRunner:
    """
    Runs all systems against all queries.
    Writes results to JSONL immediately — safe to interrupt and resume.
    Already-completed (query_id, system) pairs are skipped on resume.
    """

    def __init__(
        self,
        systems: Dict,           # {'name': BaseRAG_instance}
        dataset: List[dict],     # [{'id', 'query', 'answer', 'relevant_chunk_ids'}]
        run_id: str = None,
        max_queries: Optional[int] = None,
        log_every: int = 50
    ):
        self.systems = systems
        self.dataset = dataset[:max_queries] if max_queries else dataset
        self.run_id = run_id or f'run_{int(time.time())}'
        self.log_every = log_every
        self.out_path = f'results/{self.run_id}.jsonl'
        print(f'ExperimentRunner ready — output: {self.out_path}')

    def _already_run(self, query_id: str, system_name: str) -> bool:
        if not Path(self.out_path).exists():
            return False
        with open(self.out_path) as f:
            for line in f:
                row = json.loads(line)
                if row['query_id'] == query_id and row['system'] == system_name:
                    return True
        return False

    def _write(self, row: dict):
        with open(self.out_path, 'a') as f:
            f.write(json.dumps(row) + '\n')

    def run(self) -> pd.DataFrame:
        total = len(self.systems) * len(self.dataset)
        done  = 0

        for sys_name, system in self.systems.items():
            print(f'\n--- Running {sys_name} ({len(self.dataset)} queries) ---')
            for item in self.dataset:
                qid = item['id']
                if self._already_run(qid, sys_name):
                    done += 1
                    continue

                try:
                    result = system.query(
                        query=item['query'],
                        query_id=qid,
                        ground_truth=item.get('answer'),
                        t_now=item.get('t_now')
                    )
                    retrieved_ids = [c.id for c in result.retrieved_chunks]
                    relevant_ids  = item.get('relevant_chunk_ids', [])

                    row = {
                        'run_id':        self.run_id,
                        'system':        sys_name,
                        'query_id':      qid,
                        'query':         item['query'],
                        'ground_truth':  item.get('answer', ''),
                        'final_answer':  result.final_answer,
                        'parametric_answer': result.parametric_answer,
                        'retrieved_chunk_ids': retrieved_ids,
                        'retrieval_scores':    result.retrieval_scores,
                        'em':            exact_match(result.final_answer, item.get('answer', '')),
                        'f1':            token_f1(result.final_answer, item.get('answer', '')),
                        'recall_at_k':   recall_at_k(retrieved_ids, relevant_ids),
                        'precision_at_k': precision_at_k(retrieved_ids, relevant_ids),
                        'latency_ms':    result.latency_ms,
                        'tokens_used':   result.tokens_used,
                        't_now':         item.get('t_now'),
                    }
                    self._write(row)

                except Exception as e:
                    print(f'  ERROR {qid}: {e}')

                done += 1
                if done % self.log_every == 0:
                    print(f'  {done}/{total} complete')

        print(f'\nDone. Results in {self.out_path}')
        return self.load_results()

    def load_results(self) -> pd.DataFrame:
        rows = []
        with open(self.out_path) as f:
            for line in f:
                rows.append(json.loads(line))
        return pd.DataFrame(rows)

print('ExperimentRunner ✓')

## 4. Results aggregator

In [ ]:
class ResultsAggregator:
    """Load JSONL results and produce tables and figures."""

    def __init__(self, jsonl_path: str):
        rows = []
        with open(jsonl_path) as f:
            for line in f:
                rows.append(json.loads(line))
        self.df = pd.DataFrame(rows)
        print(f'Loaded {len(self.df)} rows from {jsonl_path}')

    def primary_table(self) -> pd.DataFrame:
        """Table 1: system × metric aggregated across all queries."""
        return (
            self.df.groupby('system')[['em', 'f1', 'recall_at_k', 'precision_at_k']]
            .agg(['mean', 'std'])
            .round(4)
        )

    def significance_test(self, system_a: str, system_b: str,
                          metric: str = 'em') -> dict:
        """Paired Wilcoxon signed-rank test between two systems."""
        df_a = self.df[self.df['system'] == system_a].set_index('query_id')[metric]
        df_b = self.df[self.df['system'] == system_b].set_index('query_id')[metric]
        common = df_a.index.intersection(df_b.index)
        a, b = df_a[common].values, df_b[common].values
        if len(set(a - b)) == 1:  # identical — Wilcoxon undefined
            return {'statistic': 0, 'p_value': 1.0, 'significant': False}
        stat, p = stats.wilcoxon(a, b)
        return {'statistic': stat, 'p_value': round(p, 4), 'significant': p < 0.05}

    def temporal_plot(self, save_path: str = 'results/temporal_degradation.png'):
        """Line plot: EM vs t_now, per system."""
        if 't_now' not in self.df.columns or self.df['t_now'].isna().all():
            print('No temporal data found — skipping plot.')
            return
        fig, ax = plt.subplots(figsize=(8, 4))
        for sys_name, grp in self.df.groupby('system'):
            by_t = grp.groupby('t_now')['em'].mean()
            ax.plot(by_t.index, by_t.values, marker='o', label=sys_name)
        ax.set_xlabel('t_now (unix timestamp)')
        ax.set_ylabel('Exact Match')
        ax.set_title('Temporal degradation of EM over time')
        ax.legend()
        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        print(f'Saved {save_path}')
        plt.show()

print('ResultsAggregator ✓')

## 5. Smoke test — mini experiment with toy data

In [ ]:
# NOTE: Run this cell only after running EbbRAG_01_core and EbbRAG_02_baselines
# (or re-define EbbRAG, VanillaRAG, BM25RAG, Chunk, ChunkIndex, Embedder, Generator above)

toy_chunks = [
    Chunk(id='c1', text='Hermann Ebbinghaus was a German psychologist who studied memory.'),
    Chunk(id='c2', text='The forgetting curve shows how memory retention decreases over time.'),
    Chunk(id='c3', text='Spaced repetition exploits the spacing effect to improve memory.'),
]
index = ChunkIndex()
index.add_chunks(toy_chunks, embedder)

toy_dataset = [
    {'id': 'q1', 'query': 'Who studied the forgetting curve?',
     'answer': 'Ebbinghaus', 'relevant_chunk_ids': ['c1']},
    {'id': 'q2', 'query': 'What does the forgetting curve show?',
     'answer': 'memory retention decreases', 'relevant_chunk_ids': ['c2']},
    {'id': 'q3', 'query': 'What technique exploits the spacing effect?',
     'answer': 'spaced repetition', 'relevant_chunk_ids': ['c3']},
]

systems = {
    'EbbRAG':     EbbRAG(index=index, embedder=embedder, generator=generator, k=2),
    'VanillaRAG': VanillaRAG(index=index, embedder=embedder, generator=generator, k=2),
    'BM25RAG':    BM25RAG(chunks=toy_chunks, generator=generator, k=2),
}

runner = ExperimentRunner(systems=systems, dataset=toy_dataset, run_id='smoke_eval')
df = runner.run()

agg = ResultsAggregator('results/smoke_eval.jsonl')
print('\nPrimary results table:')
print(agg.primary_table().to_string())

sig = agg.significance_test('EbbRAG', 'VanillaRAG', metric='em')
print(f'\nEbbRAG vs VanillaRAG significance: {sig}')
print('\nEvaluation smoke test complete ✓')